In [5]:
import pyspark 
import pandas as pd
from pyspark.sql import SparkSession,Row,DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *


spark=SparkSession.builder.master("local[1]")\
        .appName("Netflix")\
        .config("spark.driver.extraClassPath","/Users/eduardoalberto/opt/spark-4.0.0/jars/mysql-connector-j-9.1.0.jar" ) \
        .config("spark.sql.warehouse.dir", "/home/jovyan/work/database") \
        .enableHiveSupport()\
        .getOrCreate()
sc = spark.sparkContext
spark.sparkContext.setLogLevel("OFF") 
print('PySpark Version :'+spark.version)
print('PySpark Version :'+spark.sparkContext.version)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/24 23:04:21 WARN Utils: Your hostname, MacBook-Pro-de-Eduardo.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.108 instead (on interface en8)
25/08/24 23:04:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/24 23:04:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark Version :4.0.0
PySpark Version :4.0.0


In [6]:
spark.read.csv('/Users/eduardoalberto/LoadFile/input/netflix_titles_clean.csv',header=True,inferSchema=True,sep="," ,quote= '"',escape='"')\
            .createOrReplaceTempView("tb_netflix_titles")

In [ ]:
# !pip install --upgrade "pandas>=2.0.0"

#!pip install --upgrade pyspark==4.0.0
!pip3.9 list


In [35]:
# spark.table("tb_netflix_titles").filter(F.col("date_added") == "Toni Tones").show()


df = (
    spark.table("tb_netflix_titles")
        .withColumn("id", F.substring(F.col("show_id"), 2, 4))
        .withColumn("date_added", F.trim(F.col("date_added")))
        # só converte valores que correspondem ao padrão de data
        .withColumn(
            "dt_added",
            F.when(
                F.col("date_added").rlike("^[A-Za-z]+ [0-9]{1,2}, [0-9]{4}$"),
                F.to_date(F.col("date_added"), "MMMM d, yyyy")
            ).otherwise(None)
        )
)




df.write.mode("overwrite").parquet("/Users/eduardoalberto/LoadFile/repository/teste/")
df.toPandas()




,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,id,dt_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",1,2021-09-25
1,s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2,2021-09-24
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,3,2021-09-24
3,s4,TV Show,Jailbirds New Orleans,None,None,None,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",4,2021-09-24
4,s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,5,2021-09-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8804,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",8803,2019-11-20
8805,s8804,TV Show,Zombie Dumb,None,None,None,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...",8804,2019-07-01
8806,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,8805,2019-11-01
8807,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",8806,2020-01-11


In [ ]:
spark.table("tb_netflix_titles").select("rating").distinct().toPandas()

,rating
0,"November 1, 2020"
1,Shavidee Trotter
2,Adriane Lenox
3,TV-Y
4,Maury Chaykin
5,2019
6,2017
7,UR
8,Keppy Ekpenyong Bassey
9,Benn Northover


In [4]:
! ls /Users/eduardoalberto/LoadFile/output/netflix/processados/netflix/processed/

_SUCCESS
part-00000-80d25b62-9327-4b65-87c3-7dd226f9ba46-c000.snappy.parquet


In [9]:
df = spark.read.parquet("/Users/eduardoalberto/LoadFile/output/netflix/processados/netflix/processed/")
df.toPandas()



,id,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,dt_added,dt_processamento,country_name,total_shows,data_execucao
0,8,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",2021-09-24,2025-08-20 23:55:42.878845,"United States, Ghana, Burkina Faso, United Kin...",1,2025-08-20 23:55:42.878845
1,9,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...,2021-09-24,2025-08-20 23:55:42.878845,United Kingdom,1,2025-08-20 23:55:42.878845
2,10,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris ODowd, Kevin Kline, Ti...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...,2021-09-24,2025-08-20 23:55:42.878845,United States,1,2025-08-20 23:55:42.878845
3,13,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic","September 23, 2021",2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...,2021-09-23,2025-08-20 23:55:42.878845,"Germany, Czech Republic",1,2025-08-20 23:55:42.878845
4,25,s25,Movie,Jeans,S. Shankar,"Prashanth, Aishwarya Rai Bachchan, Sri Lakshmi...",India,"September 21, 2021",1998,TV-14,166 min,"Comedies, International Movies, Romantic Movies",When the father of the man she loves insists t...,2021-09-21,2025-08-20 23:55:42.878845,India,1,2025-08-20 23:55:42.878845
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5332,8802,s8802,Movie,Zinzana,Majid Al Ansari,"Ali Suliman, Saleh Bakri, Yasa, Ali Al-Jabri, ...","United Arab Emirates, Jordan","March 9, 2016",2015,TV-MA,96 min,"Dramas, International Movies, Thrillers",Recovering alcoholic Talal wakes up inside a s...,2016-03-09,2025-08-20 23:55:42.878845,"United Arab Emirates, Jordan",1,2025-08-20 23:55:42.878845
5333,8803,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",2019-11-20,2025-08-20 23:55:42.878845,United States,1,2025-08-20 23:55:42.878845
5334,8805,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,2019-11-01,2025-08-20 23:55:42.878845,United States,1,2025-08-20 23:55:42.878845
5335,8806,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",2020-01-11,2025-08-20 23:55:42.878845,United States,1,2025-08-20 23:55:42.878845
